In [ ]:
from PlacePulseDataset import PlacePulseDataset

In [ ]:
# Loads the dataset by downloading, extracting, preprocessing, and deleting the archive.
# Depending on your connection speed, might take some time to finish.

PlacePulseDataset.load()

In [ ]:
dataset = PlacePulseDataset(study_type='beautiful')

In [ ]:
from PIL import Image
(image_array, score) = dataset[0]
image = Image.fromarray(image_array)
image

In [ ]:
type(score)

In [ ]:
print(f"TrueSkill score: {score}")

# Training a model

In [ ]:
from PlacePulseDataset import transform_transformer
from torch.utils.data import Subset

train_dataset = PlacePulseDataset(study_type='beautiful', transform=transform_transformer, split='train')
val_dataset = PlacePulseDataset(study_type='beautiful', transform=transform_transformer, split='val')

indices = list(range(10))
train_dataset = Subset(train_dataset, indices)
val_dataset = Subset(val_dataset, indices)


In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

model = models.vgg16()

# Modify the classifier for linear regression
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_features, 1)  # Single output node for regression

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.classifier[6].parameters(), lr=0.001)
num_epochs = 10

for epoch in range(num_epochs):
    print(f'Epoch [{epoch+1}/{num_epochs}]')

    # Training
    train_loss = 0.0
    for images, ratings in tqdm(train_loader):
        optimizer.zero_grad()
        outputs = model(images)
        outputs = outputs.squeeze()
        loss = criterion(outputs, ratings)
        
        loss.backward()
        optimizer.step()

        train_loss += loss.item() 
    train_loss /= len(train_loader)
    print(f'Train Loss: {train_loss:.4f}')
    
    # Validation
    val_loss = 0.0
    for images, ratings in tqdm(val_loader):
        outputs = model(images)
        outputs = outputs.squeeze()
        loss = criterion(outputs, ratings)
        
        val_loss += loss.item()
    val_loss /= len(val_loader)
    print(f'Validation Loss: {val_loss:.4f}')